In [1]:
from google.colab import files

uploaded = files.upload()
print(list(uploaded))

Saving requirements.txt to requirements.txt
['requirements.txt']


In [2]:
import os
if os.path.exists("requirements.txt"):
    os.replace("requirements.txt", "requirements.txt")

!pip install -q -r requirements.txt
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 1. prepare_data.py

In [3]:
%%writefile prepare_data.py
"""Скачать страницы lib.ru и нарезать их на рассказы в JSONL.

Каркас уже качает страницы и пишет JSONL. Хармс — одна страница, рассказы
между <h2>. Чехов — десять томов с az.lib.ru, рассказ начинается с якоря.
Вам нужно дописать три места, помеченные TODO: кодировку, склейку переносов
и отбор рассказов.
"""

from __future__ import annotations

import argparse
import json
import re
import subprocess
from pathlib import Path

SOURCES = {
    "harms": ["https://www.lib.ru/HARMS/harms.txt"],
    # Тома 1–10 полного собрания. text_0032 — подписи к рисункам, не проза.
    "chekhov": [
        f"http://az.lib.ru/c/chehow_a_p/text_{number:04d}.shtml"
        for number in (10, 20, 30, 40, 50, 60, 70, 80, 90, 100)
    ],
}

# TODO: у авторов разные кодировки. Хармс на lib.ru — KOI8-R.
# Тома Чехова на az.lib.ru — windows-1251. Подставьте имена кодировок.
ENCODINGS = {"harms": "koi8-r", "chekhov": "windows-1251"}

H2_RE = re.compile(r"<h2>(.*?)</h2>", re.IGNORECASE | re.DOTALL)
ANCHOR_RE = re.compile(r'<a name="(\d+)"></a>', re.IGNORECASE)
TITLE_RE = re.compile(r"<FONT color=darkred><B>(.*?)</B>", re.IGNORECASE | re.DOTALL)
H4_TITLE_RE = re.compile(r"<h4>.*?<b>(.*?)</b>", re.IGNORECASE | re.DOTALL)
TAG_RE = re.compile(r"<[^>]+>")
MIN_CHARS = 200


def download(url: str, encoding: str) -> str:
    # curl -4: у lib.ru есть AAAA, а маршрута IPv6 часто нет.
    completed = subprocess.run(
        ["curl", "-4", "-fsSL", "--retry", "3", "--max-time", "180", url],
        check=True,
        capture_output=True,
    )
    if not encoding:
        raise NotImplementedError("укажите кодировку страницы в ENCODINGS")
    return completed.stdout.decode(encoding, errors="replace")


def html_to_text(fragment: str) -> str:
    text = TAG_RE.sub("", fragment)
    text = text.replace("&quot;", '"').replace("&lt;", "<").replace("&gt;", ">")
    text = text.replace("\xa0", " ")
    # TODO: в конце строки lib.ru часто стоит перенос слова: "изобрета-\nтельностью".
    # Замените такой перенос на склейку букв (без дефиса и без перевода строки).
    # Обычные строки ниже уже склеиваются пробелом.
    text = re.sub(r"(?<=[а-яёА-ЯЁ])-[ \t]*\n[ \t]*(?=[а-яёА-ЯЁ])", "", text)
    text = re.sub(r"[ \t]*\n[ \t]*", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


def keep_story(title: str, text: str) -> bool:
    """Решите, класть ли этот кусок в обучение.

    В страницы входят титул, оглавление, «От редакции», примечания и письма.
    Для адаптера нужны сами рассказы. Регистр сравнивайте через title.casefold():
    флаг re.IGNORECASE здесь не меняет кириллицу. Обрывки короче MIN_CHARS отбросьте.
    """
     # TODO: верните True только для рассказа.
    if len(text) < MIN_CHARS:
        return False
    lowered = title.casefold()
    not_story = (
        "оглавление",
        "содержание",
        "от редакции",
        "от издателя",
        "предисловие",
        "примечани",
        "комментари",
        "письма",
        "титул",
        "список",
    )
    return not any(word in lowered for word in not_story)
    raise NotImplementedError("опишите, какой фрагмент считается рассказом")


def split_by_h2(page: str) -> list[dict]:
    """Куски до первого <h2> отбрасываются. Так нарезана страница Хармса."""
    parts = H2_RE.split(page)
    stories = []
    for title_html, body_html in zip(parts[1::2], parts[2::2]):
        title = html_to_text(title_html)
        text = html_to_text(body_html)
        if keep_story(title, text):
            stories.append({"title": title, "text": text})
    return stories


def split_by_anchor(page: str) -> list[dict]:
    """Тома Чехова: каждый рассказ начинается с <a name="N"> и заголовка."""
    parts = ANCHOR_RE.split(page)
    stories = []
    for body_html in parts[2::2]:
        title_match = TITLE_RE.search(body_html) or H4_TITLE_RE.search(body_html)
        if title_match is None:
            continue
        title = html_to_text(title_match.group(1))
        text = html_to_text(body_html)
        if keep_story(title, text):
            stories.append({"title": title, "text": text})
    return stories


def split_stories(page: str) -> list[dict]:
    by_heading = split_by_h2(page)
    by_anchor = split_by_anchor(page)
    if len(by_anchor) >= len(by_heading) and by_anchor:
        return by_anchor
    return by_heading


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--author", choices=sorted(SOURCES), required=True)
    parser.add_argument("--out", type=Path, required=True)
    args = parser.parse_args()

    stories = []
    seen_titles = set()
    for url in SOURCES[args.author]:
        print(f"качаю {url}", flush=True)
        page = download(url, ENCODINGS[args.author])
        for story in split_stories(page):
            key = story["title"].casefold()
            if key in seen_titles:
                continue
            seen_titles.add(key)
            stories.append(story)
    if not stories:
        raise SystemExit("после чистки не осталось рассказов, проверьте кодировку и нарезку")

    args.out.parent.mkdir(parents=True, exist_ok=True)
    with args.out.open("w", encoding="utf-8") as fh:
        for story in stories:
            fh.write(json.dumps(story, ensure_ascii=False) + "\n")
    chars = sum(len(s["text"]) for s in stories)
    print(f"{args.author}: {len(stories)} рассказов, {chars} символов -> {args.out}")


if __name__ == "__main__":
    main()

Writing prepare_data.py


## 2. train_lora.py

In [4]:
%%writefile train_lora.py
"""Обучение LoRA. Цикл Trainer уже собран.

Вам нужно описать сам адаптер и то, какой пример видит модель:
реплика «напиши рассказ» и текст из корпуса. Базовые веса замораживает PEFT,
отдельно это делать не нужно.
"""

from __future__ import annotations

import os

# httpx не принимает схему socks:// (её часто ставит локальный прокси).
# До импорта huggingface её нужно заменить на socks5://.
for _key, _value in list(os.environ.items()):
    if _key.lower().endswith("proxy") and isinstance(_value, str) and _value.startswith("socks://"):
        os.environ[_key] = "socks5://" + _value[len("socks://") :]

import argparse
import json
from pathlib import Path

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)


def make_messages(story: str) -> list[dict]:
    """Соберите диалог, на котором учится адаптер.

    Генерация в generate.py использует тот же вид диалога: одна реплика user,
    без системного промпта. Не добавляйте здесь роль system.
    """
    # TODO: верните список сообщений ролей user и assistant.
    # user просит короткий рассказ, assistant — это аргумент story.
    return [
        {"role": "user", "content": "Напиши короткий рассказ."},
        {"role": "assistant", "content": story},
    ]
    raise NotImplementedError("соберите обучающий диалог")

def build_lora() -> LoraConfig:
    """Опишите LoRA. Оба автора обучаются с этой одной конфигурацией."""
    # TODO: task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05.
    # target_modules — проекции attention и MLP модели Qwen:
    # q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj.
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )
    raise NotImplementedError("соберите LoraConfig")


def load_stories(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


class StoryDataset(torch.utils.data.Dataset):
    def __init__(self, rows: list[dict], tokenizer, max_length: int):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict:
        messages = make_messages(self.rows[index]["text"])
        full = list(
            self.tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=False, return_dict=False
            )
        )[: self.max_length]
        prompt = self.tokenizer.apply_chat_template(
            messages[:1], tokenize=True, add_generation_prompt=True, return_dict=False
        )
        prompt_len = min(len(prompt), len(full))
        label = full.copy()
        # TODO: в label замените токены реплики пользователя на -100,
        # чтобы loss считался только по словам рассказа.
        # Срез уже посчитан: это первые prompt_len позиций.
        label[:prompt_len] = [-100] * prompt_len
        return {
            "input_ids": full,
            "labels": label,
            "attention_mask": [1] * len(full),
        }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--epochs", type=float, default=3)
    parser.add_argument("--max-length", type=int, default=768)
    args = parser.parse_args()

    tokenizer = AutoTokenizer.from_pretrained(args.model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=dtype)
    model.config.use_cache = False
    model = get_peft_model(model, build_lora())
    model.print_trainable_parameters()

    stories = load_stories(args.data)
    tokenized = StoryDataset(stories, tokenizer, args.max_length)
    collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, label_pad_token_id=-100)
    train_args = TrainingArguments(
        output_dir=str(args.out / "checkpoints"),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        remove_unused_columns=False,
    )
    Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=collator).train()
    model.save_pretrained(args.out)
    tokenizer.save_pretrained(args.out)
    print(f"адаптер сохранён в {args.out}")


if __name__ == "__main__":
    main()

Writing train_lora.py


## 3. generate.py

In [5]:
%%writefile generate.py
"""Сравнение трёх прогонов одним промптом: база, адаптер Хармса, адаптер Чехова.

База уже загружается в main. Допишите тему и подключение адаптера.
Параметры генерации общие для всех трёх вызовов.
"""

from __future__ import annotations

import os

# httpx не принимает схему socks:// (её часто ставит локальный прокси).
# До импорта huggingface её нужно заменить на socks5://.
for _key, _value in list(os.environ.items()):
    if (
        _key.lower().endswith("proxy")
        and isinstance(_value, str)
        and _value.startswith("socks://")
    ):
        os.environ[_key] = "socks5://" + _value[len("socks://") :]

import argparse
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen2.5-1.5B-Instruct"

# TODO: замените текст на свой бытовой сюжет. Начните фразу с «Напиши рассказ о ».
PROMPT = "Напиши рассказ о том, как человек стоит в очереди в поликлинике."

RUNS = [
    ("Базовая модель", None),
    ("LoRA: Хармс", Path("../output/harms")),
    ("LoRA: Чехов", Path("../output/chekhov")),
]


def attach_adapter(model, adapter_dir: Path):
    """Наденьте сохранённый LoRA на уже загруженную базу.

    Базу заново не загружайте: адаптер должен лечь поверх тех же весов.
    """
    # TODO: PeftModel.from_pretrained(model, adapter_dir)
    return PeftModel.from_pretrained(model, adapter_dir)


def generate(
    model, tokenizer, prompt: str, temperature: float, num_predict: int
) -> str:
    # Тот же вид диалога, что в train_lora.make_messages: одна реплика user, без system.
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=False
    )
    input_ids = torch.tensor([input_ids], device=model.device)
    attention_mask = torch.ones_like(input_ids)
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=num_predict,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output[0, input_ids.shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default=BASE)
    parser.add_argument("--out", type=Path, default=Path("generations.md"))
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--num-predict", type=int, default=400)
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    if not PROMPT.startswith("Напиши рассказ о "):
        raise SystemExit("PROMPT должен начинаться с «Напиши рассказ о »")
    if len(RUNS) != 3:
        raise SystemExit("в RUNS нужны ровно три прогона")

    tokenizer = AutoTokenizer.from_pretrained(args.model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=dtype)
    if torch.cuda.is_available():
        model = model.cuda()
    model.eval()

    lines = [f"Промпт: {PROMPT}", ""]
    for label, adapter in RUNS:
        torch.manual_seed(args.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(args.seed)
        current = model if adapter is None else attach_adapter(model, adapter)
        print(f"=== {label} ===", flush=True)
        text = generate(current, tokenizer, PROMPT, args.temperature, args.num_predict)
        lines += [f"### {label}", text, ""]
        if adapter is not None:
            model = current.unload()
    args.out.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")
    print(f"записано {args.out}")


if __name__ == "__main__":
    main()

Writing generate.py


## 4. Подготовка данных

In [6]:
!python prepare_data.py --author harms --out ../data/harms.jsonl

качаю https://www.lib.ru/HARMS/harms.txt
harms: 118 рассказов, 238074 символов -> ../data/harms.jsonl


In [7]:
!python prepare_data.py --author chekhov --out ../data/chekhov.jsonl

качаю http://az.lib.ru/c/chehow_a_p/text_0010.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0020.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0030.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0040.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0050.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0060.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0070.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0080.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0090.shtml
качаю http://az.lib.ru/c/chehow_a_p/text_0100.shtml
chekhov: 573 рассказов, 7557958 символов -> ../data/chekhov.jsonl


## 5. Обучение адаптеров

In [8]:
!python train_lora.py --data ../data/harms.jsonl --out ../output/harms

config.json: 100% 660/660 [00:00<00:00, 3.62MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 27.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 122MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 119MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 156MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 70.9M/3.09G [00:02<00:53, 56.3MB/s, 4.35MB/s  ]
model.safetensors: downloading bytes:   6% 197M/3.09G [00:03<00:26, 111MB/s, 16.4MB/s  ]
model.safetensors: downloading bytes:   7% 219M/3.09G [00:03<00:22, 128MB/s, 17.4MB/s  ]
model.safetensors: downloading bytes:   8% 252M/3.09G [00:03<00:17, 166MB/s, 19.3MB/s  ]
model.safetensors: reconstructing file:   3% 80.5M/3.09G [00:03<00:47, 63.0MB/s,  783kB/s  ]
model.safetensors: downloading bytes:   9% 275M/3.09G [00:03<00:19, 147MB/s, 23.8MB/s  ]
model.safetensors: reconstructing file:   7% 215M/

In [9]:
!python train_lora.py --data ../data/chekhov.jsonl --out ../output/chekhov

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 14061.03it/s]
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
{'loss': '2.298', 'grad_norm': '0.3639', 'learning_rate': '0.0001963', 'epoch': '0.06981'}
{'loss': '2.247', 'grad_norm': '0.3864', 'learning_rate': '0.0001917', 'epoch': '0.1396'}
{'loss': '2.245', 'grad_norm': '0.3895', 'learning_rate': '0.000187', 'epoch': '0.2094'}
{'loss': '2.159', 'grad_norm': '0.3817', 'learning_rate': '0.0001824', 'epoch': '0.2792'}
{'loss': '2.192', 'grad_norm': '0.3789', 'learning_rate': '0.0001778', 'epoch': '0.349'}
{'loss': '2.193', 'grad_norm': '0.3785', 'learning_rate': '0.0001731', 'epoch': '0.4188'}
{'loss': '2.131', 'grad_norm': '0.3788', 'learning_rate': '0.0001685', 'epoch': '0.4887'}
 17% 36/216 [12:43<1:03:36, 21.20s/it][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (149528 > 131072)

## 6. Генерация

In [10]:
!python generate.py --out generations.md

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 4770.10it/s]
=== Базовая модель ===
=== LoRA: Хармс ===
=== LoRA: Чехов ===
записано generations.md


In [11]:
print(open("generations.md", encoding="utf-8").read())

Промпт: Напиши рассказ о том, как человек стоит в очереди в поликлинике.

### Базовая модель
Вот пример рассказа о том, как человек стоит в очереди в поликлинику:

Жил-был молодой человек по имени Иван. Он решил обратиться за медицинской помощью и отправился в больницу на улице Трудовой. Встретив там огромный стройный дежурного с блестящими бирюзовыми очками и розовыми щипцами, он понял, что очередь для приема медиков начинается уже через 10 минут.

Иван долго стоял, перебирая календари, но ни один из них не показывал нужную дату. Опять же, он прислушивался к шумам вокруг: гудки телефонов, голоса врачей и пациентов, все это добавляло ему непривычной тревоги. Наконец, когда дверь подъезда начала раскрываться, Иван, наконец, собрался и пошел внутрь. 

Он прошел мимо длинных рядов столов, где стояли люди с разнообразными проблемами. Молодой человек заметил, что некоторые из них пили кофе, другие — поглощали конфеты, а еще кто-то просто смотрел телевизор. 

Пока Иван стоял, его внимание бы

In [12]:
from google.colab import files
files.download("generations.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Вывод

**Хармс отличается от базы.** База пишет нейтральный пересказ с зачином «Жил-был молодой человек по имени Иван», длинными описательными фразами и без прямой речи. Адаптер Хармса сразу начинает с реплики, пишет короткими фразами и держит действие на одном месте, сюжет не развивается.

**Чехов отличается от базы.** Текст оформлен как книжный: заголовок капсом, диалог через «--», врач и дачники как бытовая деталь из прозы автора, вопросы и возражения поочерёдно. Ирония и разговорная интонация есть, темп медленнее, чем у Хармса.


В тексте Чехова торчат nbsp; nbsp; — адаптер выучил разметку страницы, так как при чистке не была заменена эта сущность.


Русский язык не потерян, авторы не смешались, адаптеры отличаются друг от друга и от базы.

